# Speeding up Python with Numba — a hands-on demo

In this notebook we take a single, easy-to-understand task — **multiplying two matrices** (plain linear-algebra matrix multiplication) — and accelerate it step by step.

We will go through five versions, each one a rung on a *difficulty/reward ladder*:

| Version | Tool | Runs on |
|--------:|------|---------|
| 0 | NumPy `@` operator (baseline) | CPU (optimized BLAS) |
| 1 | Numba `@njit` | CPU, sequential |
| 2 | Numba `@njit(parallel=True)` + `prange` | CPU, parallel |
| 3 | Numba `@cuda.jit` | GPU |
| 4 | Numba `@cuda.jit` + explicit device memory | GPU |

The goal is to make two things concrete:
1. **The acceleration workflow** — *analyze → design → implement → evaluate (runtime + correctness)*.
2. **How to actually use Numba** — the decorators, the compilation/warm-up behavior, and the GPU memory model.

> This notebook is the hands-on companion to the Numba theory slides. The slides explain *why* and *how* Numba works; here we *use* it. Numba reference: 0.61+ with the separate **`numba-cuda`** package for the GPU target.

---

## Environment check (Colab)

Colab already ships a working, internally-consistent CUDA stack — a recent `numba` plus a matching `numba-cuda` built for Colab's CUDA 12 driver and toolkit. **Use those defaults; don't pin old versions.** Forcing year-old `numba`/`numba-cuda` releases onto a current Colab toolkit causes hard-to-debug native crashes (segmentation faults on kernel launch).

Just confirm the GPU target is available before the GPU sections. If you ever need to reset a broken environment, use **Runtime → Disconnect and delete runtime** (a plain *Restart session* keeps any bad `pip install` on disk).

In [1]:
# No installs needed — use Colab's default CUDA stack. Just confirm it's available.
import numba
from numba import cuda
print('numba', numba.__version__)
print('CUDA available:', cuda.is_available())

numba 0.60.0
CUDA available: True


## Setup: generate the input data

First we generate two random integer matrices and save them to disk, so every version reads the **same** input and we can fairly compare results and timings.

- `A` is `1000 x 1500`, `B` is `1500 x 2000`, so the result `C = A @ B` is `1000 x 2000`.
- We use small integer values to keep the numbers exact and avoid any floating-point / overflow surprises.

> The naive triple-loop versions below perform `1000 x 2000 x 1500 = 3 billion` multiply-adds. That is fine for compiled code (a few seconds) but would be hopeless in pure Python — which is exactly the point we will demonstrate.

In [2]:
import numpy as np

np.random.seed(0)  # reproducible inputs
A = np.random.randint(-100, 100, (1000, 1500))
B = np.random.randint(-100, 100, (1500, 2000))

np.savetxt('imat1.gz', A, fmt='%i')
np.savetxt('imat2.gz', B, fmt='%i')
print('A:', A.shape, '  B:', B.shape, '  -> C:', (A.shape[0], B.shape[1]))

A: (1000, 1500)   B: (1500, 2000)   -> C: (1000, 2000)


In [ ]:
!ls -lh imat1.gz imat2.gz

## Motivation: why not just write a Python loop?

Before using any acceleration, let's feel the pain. A matrix multiply is conceptually three nested loops. In **pure Python** that is catastrophically slow, so we measure it on a *much smaller* matrix and extrapolate.

This is the problem Numba exists to solve: keep the readable loop, but make it run at machine-code speed.

In [ ]:
import time

# Small matrices so pure Python finishes in a reasonable time
sA = np.random.randint(-10, 10, (100, 150))
sB = np.random.randint(-10, 10, (150, 200))

def mul_pure_python(A, B):
    C = np.zeros((A.shape[0], B.shape[1]), dtype=np.int64)
    for r in range(C.shape[0]):
        for c in range(C.shape[1]):
            tmp = 0
            for i in range(A.shape[1]):
                tmp += A[r, i] * B[i, c]
            C[r, c] = tmp
    return C

start = time.time()
C_small = mul_pure_python(sA, sB)
pure_time = time.time() - start

ops_small = sA.shape[0] * sB.shape[1] * sA.shape[1]
ops_full  = A.shape[0] * B.shape[1] * A.shape[1]
print(f'Pure Python on {sA.shape} @ {sB.shape}: {pure_time:.3f} s  ({ops_small:,} mul-adds)')
print(f'Correct vs NumPy: {np.array_equal(C_small, sA @ sB)}')
print(f'Extrapolated to the full {A.shape} @ {B.shape} ({ops_full:,} mul-adds): '
      f'~{pure_time * ops_full / ops_small / 60:.1f} minutes')

That extrapolated number is why nobody writes matrix multiply as a pure Python loop. Everything below makes the *same* loop fast — or replaces it with something faster — while keeping the code readable.

---

## Version 0 (baseline): NumPy's `@` operator

### Design
The implementation is a single line, so we can skip straight to it. NumPy's `@` dispatches to a highly optimized, precompiled **BLAS** routine (multi-threaded C/Fortran under the hood). This is our reference for both **correctness** and **speed** — it is hard to beat.

### Implementation

In [3]:
%%writefile sequential1.py
import time
import numpy as np

A = np.loadtxt('imat1.gz', dtype=int)
B = np.loadtxt('imat2.gz', dtype=int)

start = time.time()
C = A @ B
end = time.time()
print(f'Processing time: {end - start} s')

np.savetxt('omat_s1.gz', C, fmt='%i')

Writing sequential1.py


### Evaluation

#### Runtime

In [4]:
!python sequential1.py

Processing time: 15.714420080184937 s


#### Correctness

This is essentially one line of trusted library code, so we treat its output `omat_s1.gz` as the **ground truth** that every later version is checked against.

---

## Version 1: sequential CPU with Numba (`@njit`)

Now we write the triple loop ourselves and let Numba compile it to machine code.

### Design
*(File reading/writing is omitted from the design for clarity.)*

- **Input:** two matrices `A`, `B`
- **Steps:**
  - Allocate memory for the result matrix `C`
  - Call a Numba-compiled function (`@njit`) that multiplies `A` by `B` and fills `C` (for every row `r` and column `c` of `C`, set `C[r, c]` to the dot product of row `r` of `A` with column `c` of `B`)
- **Output:** the result matrix `C`

### A note on the decorator
- `@njit` is the **recommended** way to compile in *nopython* mode (fully native, no Python interpreter).
- `@njit` is exactly `@jit(nopython=True)`. Since **Numba 0.59**, plain `@jit` also defaults to nopython mode, so `@jit` and `@njit` now behave the same — `@njit` just makes the intent explicit.
- `cache=True` saves the compiled code to disk (`__pycache__`) so we don't recompile on every fresh run.

### Implementation

In [ ]:
%%writefile sequential2.py
import time
import numpy as np
from numba import njit  # njit == jit(nopython=True)

@njit(cache=True)
def mul_mat(A, B, C):
    for r in range(C.shape[0]):
        for c in range(C.shape[1]):
            temp = 0
            for i in range(A.shape[1]):
                temp += A[r, i] * B[i, c]
            C[r, c] = temp

A = np.loadtxt('imat1.gz', dtype=int)
B = np.loadtxt('imat2.gz', dtype=int)

start = time.time()
C = np.empty((A.shape[0], B.shape[1]), dtype=int)
mul_mat(A, B, C)
end = time.time()
print(f'Processing time: {end - start} s')

np.savetxt('omat_s2.gz', C, fmt='%i')

### Evaluation

#### Runtime

**Run #1** is usually slower because Numba has to **compile** the function on its first call (the *warm-up cost* discussed in the theory slides). The compiled result is cached in `__pycache__`:

In [ ]:
!python sequential2.py  # Run #1 (includes compilation)

In [ ]:
!ls -a __pycache__/

**Run #2** is faster because it reuses the cached compiled code instead of recompiling:

In [ ]:
!python sequential2.py  # Run #2 (uses cache)

> **Why might Run #2 _not_ look faster?** The timer wraps the whole `mul_mat` call, and for these large matrices that call is dominated by the *actual computation* — about 15 s of work (3 billion multiply-adds, single-threaded), not by compilation. Compiling the function only costs roughly a second, so caching it away barely dents a ~15 s total, and on a shared Colab CPU the normal run-to-run variation can easily hide — or even reverse — that small saving. The cache **is** working (the `.nbc`/`.nbi` files above prove it); the compilation/warm-up cost simply becomes *visible* only when the computation itself is fast. To see it clearly, call the function twice on a tiny matrix and time each call separately: the first call pays for compilation, the second is essentially instant.

Overall, this hand-written loop ends up roughly comparable to NumPy's `@` for this task. That is already remarkable: a readable Python triple loop, compiled by Numba, competes with a tuned BLAS library — and unlike BLAS, *you can put any logic you like inside the loop*.

> ⚠️ **Benchmarking rule:** never time Run #1 and conclude "Numba is slow." The first call pays for compilation. Measure the second call, or warm the function up first.

#### Correctness

In [8]:
C_s1 = np.loadtxt('omat_s1.gz', dtype=int)

In [ ]:
C_s2 = np.loadtxt('omat_s2.gz', dtype=int)
mad = np.mean(np.abs(C_s2 - C_s1))
print('Mean abs difference vs baseline:', mad)
assert mad == 0, 'Result differs from the NumPy baseline!'
print('OK: identical to baseline')

---

## Version 2: parallel CPU with Numba (`@njit(parallel=True)` + `prange`)

Our course project focuses on *running in parallel on the GPU*. This CPU version is included mainly to show that Numba can **also** parallelize across CPU cores with almost no extra code.

### Analysis: which step should we parallelize?
This tiny program has a single data-processing step (the matrix multiply), so there is no choice — that is the step we accelerate. (In a larger program, like the course project, there are many steps; you would *time each step* to find the bottleneck and focus your effort there.) Fortunately this step parallelizes well, because **each element of the result matrix is computed independently** of the others.

### Design

In [ ]:
!lscpu | grep -E 'Model name|^CPU\(s\)|Thread|Core'

Each output element `C[r, c]` is independent, so the work can be split across threads. With Numba's CPU parallelism you only have to **tell Numba which loops are independent** (using `prange`); Numba then distributes those iterations across threads automatically — you don't manage threads yourself.

- `prange` replaces `range` on the loop(s) that can run in parallel.
- `parallel=True` enables Numba's automatic parallelization pass.
- Setting the threading layer (e.g. OpenMP) is optional; it controls which backend runs the threads.

### Implementation

In [ ]:
%%writefile parallel1.py
import time
import numpy as np
from numba import njit, prange
from numba import config
config.THREADING_LAYER = 'omp'  # use the OpenMP threading backend

@njit(parallel=True, cache=True)
def mul_mat(A, B, C):
    for r in prange(C.shape[0]):   # prange: this loop's iterations are independent
        for c in range(C.shape[1]):
            temp = 0
            for i in range(A.shape[1]):
                temp += A[r, i] * B[i, c]
            C[r, c] = temp

A = np.loadtxt('imat1.gz', dtype=int)
B = np.loadtxt('imat2.gz', dtype=int)

start = time.time()
C = np.empty((A.shape[0], B.shape[1]), dtype=int)
mul_mat(A, B, C)
end = time.time()
print(f'Processing time: {end - start} s')

np.savetxt('omat_p1.gz', C, fmt='%i')

> **Tip:** parallelize the **outer** loop (`prange` on `r`). Each thread then owns whole rows of `C`, which keeps the work per thread large and avoids threads fighting over the same cache lines. Marking the inner loop parallel too is usually unnecessary and can hurt.

### Evaluation

#### Runtime (run twice — first run compiles)

In [ ]:
!python parallel1.py  # Run #1 (includes compilation)

In [ ]:
!python parallel1.py  # Run #2 (uses cache)

Is it faster than the sequential version? The speedup is bounded by the number of available cores/threads (and by memory bandwidth). On a 2-core / 4-thread machine, expect *up to* roughly a few times faster — never the full thread count, because of overhead and shared-resource contention (**Amdahl's law** in action).

#### Correctness

In [ ]:
C_p1 = np.loadtxt('omat_p1.gz', dtype=int)
mad = np.mean(np.abs(C_p1 - C_s1))
print('Mean abs difference vs baseline:', mad)
assert mad == 0
print('OK: identical to baseline')

---

## Version 3: GPU with Numba (`@cuda.jit`)

This is the kind of parallelism the course project focuses on.

### Analysis
Same as before: the single multiply step is the only candidate, and it parallelizes well because every `C[r, c]` is independent. But the GPU has a *very different* execution model from the CPU.

### Design: the GPU thread/block/grid model
On the GPU we launch **thousands of threads**, each running the same *kernel* function. We assign **one thread per output element** `C[r, c]`.

Threads are organized in a 2-level hierarchy so they can be mapped naturally onto a 2D matrix:

```text
          GRID  (all blocks)
  +-----------+-----------+-----------+
  |  block    |  block    |  block    |
  | +-+-+-+-+ | +-+-+-+-+ |           |   each block is a 2D tile of threads
  | | | | | | | | | | | | |    ...    |   (here 32 x 32 threads per block)
  | +-+-+-+-+ | +-+-+-+-+ |           |
  +-----------+-----------+-----------+   each thread computes ONE C[r, c]
```

We choose a **2D block** and **2D grid**, mapping:
- the **x** dimension of block/grid to the matrix's **horizontal** direction (columns → `c`)
- the **y** dimension of block/grid to the matrix's **vertical** direction (rows → `r`)

A thread finds *its* `(r, c)` from its position in the grid:
```python
r = cuda.blockIdx.y * cuda.blockDim.y + cuda.threadIdx.y
c = cuda.blockIdx.x * cuda.blockDim.x + cuda.threadIdx.x
# cuda.grid(2) is a shortcut that returns (x_index, y_index) = (c, r)
```
Because we usually launch *more* threads than there are elements (the grid is rounded up), every kernel must **check its bounds** (`if r < ... and c < ...`) so extra threads do nothing.

[Illustration of the grid/block layout](https://docs.google.com/spreadsheets/d/13upl8S0wBESvKUIZVO98-f5JD6rA4Mfyvepw_bDAHoc/edit?usp=sharing)

### Implementation

The GPU target lives in the separate, NVIDIA-maintained **`numba-cuda`** package (the built-in CUDA target in core Numba is deprecated). Your code still uses the `numba.cuda` namespace, and on Colab this package is already installed — no setup needed.

In [5]:
%%writefile parallel2.py
import time
import math
import numpy as np
from numba import cuda

from numba import config
config.CUDA_ENABLE_PYNVJITLINK = 1

@cuda.jit  # add cache=True to enable on-disk caching (not used here)
def mul_mat_kernel(A, B, C):  # this is the code ONE GPU thread runs
    # r = cuda.blockIdx.y * cuda.blockDim.y + cuda.threadIdx.y
    # c = cuda.blockIdx.x * cuda.blockDim.x + cuda.threadIdx.x
    c, r = cuda.grid(2)  # shortcut for the two lines above: returns (x, y) = (c, r)

    if r < C.shape[0] and c < C.shape[1]:  # guard against the extra threads
        temp = 0
        for i in range(A.shape[1]):
            temp += A[r, i] * B[i, c]
        C[r, c] = temp

A = np.loadtxt('imat1.gz', dtype=int)
B = np.loadtxt('imat2.gz', dtype=int)

start = time.time()
C = np.empty((A.shape[0], B.shape[1]), dtype=int)

block_size = (32, 32)  # 32*32 = 1024 threads per block (a common maximum)
grid_size = (math.ceil(C.shape[1] / block_size[0]),   # x covers columns
             math.ceil(C.shape[0] / block_size[1]))   # y covers rows

mul_mat_kernel[grid_size, block_size](A, B, C)
cuda.synchronize()  # wait for the GPU to finish before stopping the timer
end = time.time()
print(f'Processing time: {end - start} s')

np.savetxt('omat_p2.gz', C, fmt='%i')

Writing parallel2.py


> **Important subtlety — implicit data transfer.** Here we pass ordinary NumPy (host) arrays straight to the kernel. Numba automatically copies `A`, `B` to the GPU before the kernel and copies `C` back afterward. That is convenient, but **the transfer cost is hidden inside the timing**, and it happens on *every* call. The next version makes the transfer explicit so we can separate it from the compute. Also note the `cuda.synchronize()`: kernel launches are *asynchronous*, so without it we might stop the timer before the GPU has finished.

### Evaluation

#### Runtime

In [6]:
!python parallel2.py

/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/cudadrv/devicearray.py:934: NumbaPerformanceWarning: Host array used in CUDA kernel will incur copy overhead to/from device.
  warn(NumbaPerformanceWarning(msg))
Processing time: 1.9448742866516113 s


On a typical Colab GPU this comes out noticeably faster than the sequential `@njit` version — often several times faster even with the naive algorithm. (For a fair comparison against `@njit`, compare against `@njit`'s **Run #1**, since these kernels don't enable caching. Note that `@cuda.jit` *can* cache with `cache=True` in modern numba-cuda — we just don't use it here, so a faster *second* GPU run reflects CUDA driver/context warm-up, not cached compilation.)

#### Correctness

In [9]:
C_p2 = np.loadtxt('omat_p2.gz', dtype=int)
mad = np.mean(np.abs(C_p2 - C_s1))
print('Mean abs difference vs baseline:', mad)
assert mad == 0
print('OK: identical to baseline')

Mean abs difference vs baseline: 0.0
OK: identical to baseline


---

## Version 4: GPU with explicit device memory management

This is the *same kernel* as Version 3, but we manage the GPU memory ourselves instead of letting Numba do it implicitly. This matters for two reasons:

1. **Honest timing.** We move the data to the device *before* starting the clock, so the timer measures the **compute**, not the one-off transfer.
2. **Realistic usage.** In a real program you transfer data to the GPU once and run *many* kernels on it, reusing the device arrays. Re-transferring every call (as the implicit version does) would waste most of your time on the PCIe bus. The data-transfer cost is often the real bottleneck — not the math.

### Implementation

Key new calls:
- `cuda.to_device(x)` — copy a host array to the GPU, returns a device array.
- `cuda.device_array(shape, dtype)` — allocate an *uninitialized* array directly on the GPU (for outputs).
- `d_C.copy_to_host(C)` — copy the GPU result back to host.
- `cuda.synchronize()` — block until the GPU has finished (needed for correct timing).

In [10]:
%%writefile parallel3.py
import time
import math
import numpy as np
from numba import cuda

from numba import config
config.CUDA_ENABLE_PYNVJITLINK = 1

@cuda.jit  # add cache=True to enable on-disk caching (not used here)
def mul_mat_kernel(A, B, C):  # the code ONE GPU thread runs
    c, r = cuda.grid(2)  # (x, y) = (c, r)
    if r < C.shape[0] and c < C.shape[1]:
        temp = 0
        for i in range(A.shape[1]):
            temp += A[r, i] * B[i, c]
        C[r, c] = temp

A = np.loadtxt('imat1.gz', dtype=int)
B = np.loadtxt('imat2.gz', dtype=int)

C = np.empty((A.shape[0], B.shape[1]), dtype=int)
block_size = (32, 32)
grid_size = (math.ceil(C.shape[1] / block_size[0]),
             math.ceil(C.shape[0] / block_size[1]))

# Move inputs to the device and allocate the output ON the device
d_A = cuda.to_device(A)
d_B = cuda.to_device(B)
d_C = cuda.device_array((A.shape[0], B.shape[1]), dtype=int)

cuda.synchronize()
start = time.time()

mul_mat_kernel[grid_size, block_size](d_A, d_B, d_C)  # compute only

cuda.synchronize()
end = time.time()

d_C.copy_to_host(C)  # bring the result back
print(f'Processing time (compute only): {end - start} s')

np.savetxt('omat_p3.gz', C, fmt='%i')

Writing parallel3.py


### Evaluation

#### Runtime

In [11]:
!python parallel3.py

Processing time (compute only): 0.5346863269805908 s


The reported time is now **compute only** — usually smaller than Version 3's number, because we excluded the host↔device transfer from the measurement. The difference between the two is a rough estimate of the **transfer overhead** you pay each time you move data across the bus.

#### Correctness

In [ ]:
C_p3 = np.loadtxt('omat_p3.gz', dtype=int)
mad = np.mean(np.abs(C_p3 - C_s1))
print('Mean abs difference vs baseline:', mad)
assert mad == 0
print('OK: identical to baseline')

---

## How to benchmark Numba honestly

Timing GPU/JIT code correctly is its own skill. Three rules that the versions above hint at, collected in one place:

1. **Warm up first.** The first call compiles. Call the function once (on tiny data or the real data) before timing.
2. **Repeat and take the best/median.** A single measurement is noisy; run several and report the minimum or median.
3. **Synchronize the GPU.** Kernel launches are asynchronous. Call `cuda.synchronize()` (or use CUDA events) *before* reading the clock, or you will time the launch instead of the work.

A small reusable helper for the **CPU** functions (we can run this here without a GPU):

In [ ]:
import time
import numpy as np
from numba import njit, prange

def benchmark(fn, *args, warmup=1, repeat=5):
    """Warm up, then time `repeat` runs; return the best (minimum) time in seconds."""
    for _ in range(warmup):
        fn(*args)
    best = float('inf')
    for _ in range(repeat):
        t0 = time.perf_counter()
        fn(*args)
        best = min(best, time.perf_counter() - t0)
    return best

# Note: no cache=True here. Caching needs a real source file; functions defined
# inside a notebook cell can't always be cached, and there's no benefit within one session.
@njit
def mul_seq(A, B, C):
    for r in range(C.shape[0]):
        for c in range(C.shape[1]):
            t = 0
            for i in range(A.shape[1]):
                t += A[r, i] * B[i, c]
            C[r, c] = t

@njit(parallel=True)
def mul_par(A, B, C):
    for r in prange(C.shape[0]):
        for c in range(C.shape[1]):
            t = 0
            for i in range(A.shape[1]):
                t += A[r, i] * B[i, c]
            C[r, c] = t

C = np.empty((A.shape[0], B.shape[1]), dtype=int)

t_numpy = benchmark(lambda: A @ B)
t_seq   = benchmark(mul_seq, A, B, C)
t_par   = benchmark(mul_par, A, B, C)

print(f'NumPy @            : {t_numpy*1e3:8.1f} ms')
print(f'Numba @njit (seq)  : {t_seq*1e3:8.1f} ms   ({t_numpy/t_seq:5.2f}x vs NumPy)')
print(f'Numba parallel     : {t_par*1e3:8.1f} ms   ({t_seq/t_par:5.2f}x vs seq)')

Fill in a summary table for your own machine. Example shape (your numbers will differ by hardware):

| Version | Runs on | Time | Speedup vs seq `@njit` |
|---------|---------|------|------------------------|
| NumPy `@` | CPU (BLAS) | … | … |
| `@njit` | CPU, 1 thread | … | 1.0x |
| `@njit(parallel=True)` | CPU, N threads | … | … |
| `@cuda.jit` (implicit mem) | GPU | … | … |
| `@cuda.jit` (explicit mem) | GPU, compute only | … | … |

---

## Bonus: `@vectorize` — turn a scalar function into a fast NumPy ufunc

`@njit` and `@cuda.jit` are the workhorses, but Numba has more decorators. `@vectorize` takes a function written for **single scalars** and compiles it into a true NumPy **ufunc** that automatically applies element-wise across whole arrays (with broadcasting), at machine-code speed — and can even target the GPU by passing `target='cuda'`.

You write the math for *one* element; Numba handles the looping and broadcasting.

In [ ]:
import numpy as np
from numba import vectorize

@vectorize(['float64(float64, float64)'])  # signature: (in, in) -> out
def hypotenuse(x, y):
    return (x*x + y*y) ** 0.5  # written for SINGLE numbers

x = np.array([3.0, 5.0, 8.0])
y = np.array([4.0, 12.0, 15.0])
print(hypotenuse(x, y))          # applied element-wise automatically
print(hypotenuse(x, 1.0))        # broadcasting works too

This is the same idea as a NumPy ufunc like `np.add`, but you define the per-element logic yourself. It is ideal when your operation is element-wise but not expressible as a simple chain of existing NumPy operations.

---

## Key takeaways & common pitfalls

**The acceleration workflow** — always *analyze* (find the costly step), *design*, *implement*, then *evaluate* both **runtime** and **correctness**. Speed is worthless if the answer is wrong.

**Numba decorators**
- `@njit` (= `@jit(nopython=True)`): compile to fast CPU machine code. Modern default.
- `@njit(parallel=True)` + `prange`: spread independent loop iterations across CPU threads.
- `@cuda.jit`: write a GPU kernel — one thread per output element, organized in blocks and a grid.
- `@vectorize`: build a fast element-wise ufunc from a scalar function.

**Pitfalls to remember**
- **Warm-up cost:** the first call compiles. Never benchmark Run #1. `cache=True` persists compilation across runs and works for **both** `@njit` and `@cuda.jit` (modern numba-cuda). A faster second GPU run *without* `cache=True` is CUDA driver/context warm-up, not caching.
- **Data-transfer overhead (GPU):** moving arrays host↔device can cost more than the computation. Transfer once, reuse device arrays, and exclude transfer from compute timing when that's what you want to measure.
- **Asynchronous kernels:** call `cuda.synchronize()` before stopping a timer, or you'll measure the launch, not the work.
- **Always bounds-check in a kernel:** the grid is rounded up, so some threads have no element to compute.
- **`cuda.grid(2)` returns `(x, y)`:** map x→column and y→row carefully (`c, r = cuda.grid(2)`); swapping them is a classic bug.
- **Naive vs library:** our hand-written matmul is for *learning*. In production, `A @ B` (BLAS) or cuBLAS/CuPy will usually beat a naive kernel. Numba shines when your computation *isn't* a standard library call.
- **`numba-cuda` packaging:** the GPU target is now a separate NVIDIA-maintained package; install it alongside Numba. Code keeps using the `numba.cuda` namespace.

---

## References

- Numba documentation — https://numba.readthedocs.io/en/stable/
- `@jit` / compilation modes — https://numba.readthedocs.io/en/stable/user/jit.html
- Parallel CPU (`prange`, `parallel=True`) — https://numba.readthedocs.io/en/stable/user/parallel.html
- Numba for CUDA GPUs — https://numba.readthedocs.io/en/stable/cuda/index.html
- `numba-cuda` (NVIDIA) — https://nvidia.github.io/numba-cuda/
- `@vectorize` / ufuncs — https://numba.readthedocs.io/en/stable/user/vectorize.html